# 🔍 01 — RAG Pipeline: İstanbul Bilgi Erişim Sistemi

Bu notebook İstanbul belgelerini vektöre dönüştürür ve FAISS indeksi oluşturur.

## 📂 Adım 1: Dataset Yükle

In [4]:
from datasets import load_from_disk

dataset = load_from_disk("rag_data/istanbul_dataset")
print(f"✅ {len(dataset)} İstanbul kategorisi yüklendi")
for i, ornek in enumerate(dataset):
    print(f"[{i:02d}] {ornek['kaynak']} — {len(ornek['metin'].split())} kelime")

✅ 11 İstanbul kategorisi yüklendi
[00] kayit_yenileme — 137 kelime
[01] sinav_sistemi — 161 kelime
[02] burslar_indirimler — 139 kelime
[03] mezuniyet_sartlari — 146 kelime
[04] ogrenci_hizmetleri — 128 kelime
[05] kampus_ve_ulasim — 116 kelime
[06] erasmus_uluslararasi — 107 kelime
[07] ois_teknik_destek — 103 kelime
[08] akademik_takvim — 138 kelime
[09] disiplin_ve_kurallar — 123 kelime
[10] yurt_ve_konaklama — 98 kelime


## ✂️ Adım 2: Metinleri Chunk'lara Böl

In [5]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

all_chunks = []
chunk_meta = []

for row in dataset:
    chunks = chunk_text(row["metin"])
    for c in chunks:
        all_chunks.append(c)
        chunk_meta.append({"kaynak": row["kaynak"]})

print(f"✅ Toplam chunk sayisi: {len(all_chunks)}")

✅ Toplam chunk sayisi: 18


## 🧠 Adım 3: Embedding Oluştur (Sentence Transformers)

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Model yüklendi: paraphrase-multilingual-MiniLM-L12-v2")

print("🔄 Embedding oluşturuluyor...")
embeddings = model.encode(all_chunks, batch_size=32, show_progress_bar=True)
embeddings = embeddings.astype("float32")
print(f"✅ Embedding boyutu: {embeddings.shape}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2669.64it/s]


✅ Model yüklendi: paraphrase-multilingual-MiniLM-L12-v2
🔄 Embedding oluşturuluyor...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.47s/it]

✅ Embedding boyutu: (18, 384)


## 📦 Adım 4: FAISS İndeksi Oluştur ve Kaydet

In [7]:
import faiss
import pickle

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"✅ FAISS indeksi oluşturuldu: {index.ntotal} vektör")

faiss.write_index(index, "rag_data/istanbul_index.faiss")
with open("rag_data/chunks.pkl", "wb") as f:
    pickle.dump({"chunks": all_chunks, "meta": chunk_meta}, f)

print("✅ Kaydedildi: rag_data/istanbul_index.faiss")
print("✅ Kaydedildi: rag_data/chunks.pkl")

✅ FAISS indeksi oluşturuldu: 18 vektör
✅ Kaydedildi: rag_data/istanbul_index.faiss
✅ Kaydedildi: rag_data/chunks.pkl


## 🧪 Adım 5: Arama Testi

In [10]:
def ara(soru, k=3):
    q_vec = model.encode([soru]).astype("float32")
    distances, indices = index.search(q_vec, k)
    print(f"🔍 Soru: {soru}")
    print("-" * 60)
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        print(f"[{rank+1}] Kaynak: {chunk_meta[idx]['kaynak']} (mesafe: {dist:.2f})")
        print(f"{all_chunks[idx][:200]}...")

ara("Ayasofya kaç yılında yapıldı?")
ara("İstanbul'dan Asya yakasına nasıl geçerim?")

🔍 Soru: Ayasofya kaç yılında yapıldı?
------------------------------------------------------------
[1] Kaynak: akademik_takvim (mesafe: 13.68)
2024-2025 Akademik Takvimi Güz Yarıyılı: - Kayıt Yenileme: 16-20 Eylül 2024 - Ders Başlangıcı: 23 Eylül 2024 - Ekle-Sil: 23 Eylül - 4 Ekim 2024 - Ara Sınavlar (Vize): 11-22 Kasım 2024 - Son Ders Günü:...
[2] Kaynak: kayit_yenileme (mesafe: 16.03)
İstanbul Topkapı Üniversitesi Ders Kayıt Yenileme Prosedürü Ders kayıt yenileme işlemleri her yarıyıl başında Öğrenci Bilgi Sistemi (OİS) üzerinden yapılmaktadır. Kayıt tarihleri akademik takvimde ila...
[3] Kaynak: burslar_indirimler (mesafe: 16.14)
Burs ve İndirim Sistemi ÖSYS Bursu: LYS/YKS sırasına göre verilen burstur. - İlk 1.000: %100 burs (tam burs) - İlk 10.000: %75 burs - İlk 25.000: %50 burs - İlk 50.000: %25 burs Burslar her yıl yenide...
🔍 Soru: İstanbul'dan Asya yakasına nasıl geçerim?
------------------------------------------------------------
[1] Kaynak: kampus_ve_ulasim (mesafe: 20.85)